# **4 Кейс**

**Описание кейса:** требуется классифицировать риск задержки доставки заказа более чем на 5 минут от обещанного времени на этапе оформления, используя параметры ресторана, рейтинг курьера, погодные условия и пробки.

https://www.kaggle.com/datasets/razanihababdellatif/food-delivery-orders-and-eta-logistics-dataset

## **Импорт библиотек**

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

## **Загрузка данных**

In [ ]:
df = pd.read_csv("food_delivery_orders_dataset.csv")

## **Первичный анализ**

In [ ]:
pd.set_option('display.max_columns', None)
df.head()

Целевой показатель - **late_delivery**  
Заказ помечается как доставленный с опозданием (late_delivery = 1.0) только в том случае, если фактическое время доставки превышает расчетное более чем на 5 минут

Явно лишние показатели:  
**order_id**, **customer_id**, **restaurant_id** - не несут смысла  
**actual_delivery_time_minutes** - фактическое время доставки не имеет значения, важен факт опоздал курьер или нет

Нужно проверить как **late_delivery** зависит от **driver_id**  

Показатели **order_date**, **order_hour**, **day_of_week**, **is_weekend** можно выразить из **order_timestamp**  
Показатель **city** можно выразить из **delivery_area**.  
Показатель **order_total** выражается из суммы показателей (**subtotal** - **discount_percent**%), **tax_amount**, **service_fee**, **delivery_fee**

In [ ]:
df['late_delivery'].value_counts()

Есть перевес в сторону записей без опоздания. Записей без опоздания больше примерно в 1.73 раза

In [ ]:
print(f"Кол-во строк: {df.shape[0]}")
print(f"Кол-во столбцов: {df.shape[1]}")

In [ ]:
df.dtypes.value_counts()

Категориальных параметров - 17  
Непрерывных параметров - 20 (12 + 8)

In [ ]:
# Не получилось вывести все колонки через df.info ))
pd.DataFrame({
    "dtype":   df.dtypes,
    "non_null": df.notna().sum(),
    "null":    df.isna().sum(),
    "null_%":  (df.isna().mean() * 100).round(2),
})

Есть несоответствие типов:  
**order_timestamp** - строка, хотя логичней тип datetime.datetime  
**order_date** - строка, хотя логичней тип datetime.date

## **Анализ непрерывных показателей**

In [ ]:
df.describe().T

Аномалий не вижу  
Скорей всего есть выбросы, если смотреть на 75 квартиль и максимальное значение параметров **order_total**, **service_fee**, **restaurant_preparation_time_minutes** и **tip_amount**. Посмотрим на этой подробней при визуализации

## **Анализ категориальных показателей**

In [ ]:
df.describe(include=["str"]).T

Аномалий не вижу

## **Анализ дублей**

In [ ]:
df.duplicated().sum()

Дублей нет

## **Анализ пропусков**

**tip_amount**, **actual_delivery_time_minutes**, **late_delivery**, **customer_rating** имеют одинаковое количество пропусков - 868  
У **cancellation_reason** 49132 пропусков, что равно 50000-868  

**late_delivery** - целевая параметр, в ней есть пропуски. Мы можем удалить эти строки, так как нам не нужны строки без ответа

Можно предположить что все пропуски в одних и тех же строках, и после очистки строк с пустым **late_delivery** пропуски пропадут  

Проверим строки с пустым **late_delivery**:

In [ ]:
df[df["late_delivery"].isna()].head(15)

Для записей у которых **late_delivery** пустой:  
**tip_amount** = NaN  
**actual_delivery_time_minutes** = NaN  
**order_status** = Cancelled  
**cancellation_reason** заполнен  
**customer_rating** = NaN  

Из этого можно сделать вывод: **customer_rating**, **actual_delivery_time_minutes** и **tip_amount** заполняются только после выполнения доставки. Для прогноза будет ли выполнена доставка без опоздания они НЕ нужны  
**cancellation_reason** заполнен только для записей у которых **late_delivery** не заполнен, для обучения нам нужны только записи с заполненным **late_delivery**, этот параметр не нужен  
**order_status** имеет 2 значения **Cancelled** (для записей **late_delivery.isna()**) и **Completed** (для записей **late_delivery.notna()**). Нас интересуют только записи с заполненным **late_delivery**, в этих записях у order_status будет 1 уникальное значение - **Completed**, можно удалить этот параметр

## **Промежуточный итог**

**Размер датасета:**  
Кол-во строк: 50000  
Кол-во столбцов: 37  

Категориальных параметров - 17  
Непрерывных параметров - 20  

Целевой показатель - **late_delivery**  
Есть перевес в сторону записей без опоздания (**late_delivery** = 0). Записей без опоздания больше примерно в 1.73 раза  

Есть несоответствие типов:  
**order_timestamp** - строка, хотя логичней тип datetime.datetime  
**order_date** - строка, хотя логичней тип datetime.date. 

**Дубликатов в датасете нет**

**Пропуски:**
**tip_amount**, **actual_delivery_time_minutes**, **customer_rating** и **late_delivery** имеют одинаковое количество пропусков - 868  
У **cancellation_reason** 49132 пропусков, что равно 50000-868. Там где **late_delivery** = NaN - **cancellation_reason** имеет значение, во всех остальных случаях **cancellation_reason** пустой  

Есть записи с пустой целевой переменной (**late_delivery** = NaN). Нужно будет удалить эти записи из Датасета
Все пропуски в одних и тех же строках, и после очистки строк с пустым **late_delivery** пропуски пропадут. Останутся пропуски в **cancellation_reason**, но об этом ниже  

На этом этапе уже можно выделить **лишние показатели**:  
**order_id**, **customer_id**, **restaurant_id** - не несут смысла для прогноза будет ли доставка выполнена без опоздания  
**actual_delivery_time_minutes** - фактическое время доставки в этой задаче не имеет значения, важен факт опоздал курьер или нет  
**customer_rating**, **actual_delivery_time_minutes** и **tip_amount** заполняются только после выполнения доставки. Для прогноза будет ли доставка выполнена без опоздания они не нужны  
**cancellation_reason** заполнен только для записей у которых **late_delivery** не заполнен, для обучения нам нужны только записи с заполненным **late_delivery**, этот параметр не нужен  
**order_status** имеет 2 значения **Cancelled** (для записей **late_delivery** = NaN) и **Completed** (для всех остальных записей). Нас интересуют только записи с заполненным **late_delivery**, в этих записях у **order_status** будет 1 уникальное значение - **Completed**, можно удалить этот параметр. 

Итого список столбцов на удаление: **order_id**, **customer_id**, **restaurant_id**, **actual_delivery_time_minutes**, **customer_rating**, **tip_amount**, **cancellation_reason**, **order_status**


**Открытые вопросы после первичного анализа:**  
1. Есть ли зависимоть **late_delivery** от **driver_id**?  
2. Показатели **order_date**, **order_hour**, **day_of_week**, **is_weekend** можно выразить из **order_timestamp**. Какие признаки оставить?  
3. Показатель **city** можно выразить из **delivery_area**. Какой признаки оставить?  
4. Показатель **order_total** выражается из суммы показателей (**subtotal** - **discount_percent**%), **tax_amount**, **service_fee**, **delivery_fee**. Какие признаки оставить?  


## **Преобразования после первичного анализа**

Удаление строк с пустым целевым показателем:

In [ ]:
df = df[df["late_delivery"].notna()]

Преобразование показателей с датой к типу datetime и date:

In [ ]:
df['order_timestamp'] = pd.to_datetime(df['order_timestamp'])
df['order_date'] = pd.to_datetime(df['order_date']).dt.date

Удаление лишних показателей:

In [ ]:
df.drop([
    "order_id",
    "customer_id",
    "restaurant_id",
    "actual_delivery_time_minutes",
    "customer_rating",
    "tip_amount",
    "cancellation_reason",
    "order_status",
], axis=1, inplace=True)

Проверка текущего размера датасета и пропусков:

In [ ]:
print(f"Кол-во строк: {df.shape[0]}")
print(f"Кол-во столбцов: {df.shape[1]}")

pd.DataFrame({
    "dtype":   df.dtypes,
    "null":    df.isna().sum()
})

# TODO
---

## **Анализ выбросов**